# Customer Churn — Feature Reduction, Retrain & Compare

**Goal:** Retrain Logistic Regression on a **reduced 25-feature** set and compare it fairly with the current **37-feature** model (96.24% accuracy) before replacing any artifacts.

**Rules used here:**
- Same split: `test_size=0.2`, `random_state=42`, `stratify=y`
- Same preprocessing: `OneHotEncoder` + `StandardScaler` + `LogisticRegression`
- Same tuning: `GridSearchCV`, `C ∈ {0.01, 0.1, 1, 10, 100}`, `cv=5`, `scoring="f1"`
- Artifacts are **backed up** and only overwritten if the reduced model is acceptable

Run cells **top to bottom**. Do not modify FastAPI/React in this notebook.

## 1 — Setup

In [ ]:
# Pin to the same sklearn version as backend/requirements.txt (1.6.1)
!pip install -q "scikit-learn==1.6.1" "pandas==2.2.3" "numpy==2.1.3" "joblib==1.6.0"

In [ ]:
import os
import shutil
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

print("sklearn setup OK")

## 2 — Load dataset

Upload your Telco Customer Churn CSV (the one used to train the current model). Expected target: `Churn Value` (0/1) or `Churn Label` (Yes/No) — auto-detected below.

In [ ]:
# Option A: Google Colab upload dialog
try:
    from google.colab import files

    print("Select your churn CSV (e.g. Telco-Customer-Churn.csv)…")
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded))
except Exception:
    # Option B: local / Drive path — edit if needed
    DATA_PATH = "Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}  shape={df.shape}")
df.head(3)

In [ ]:
# Detect target column
TARGET_CANDIDATES = ["Churn Value", "Churn Label", "Churn", "churn"]
TARGET = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
if TARGET is None:
    raise KeyError(f"No churn target found among {TARGET_CANDIDATES}. Columns: {list(df.columns)}")

if df[TARGET].dtype == object:
    y = df[TARGET].map({"Yes": 1, "No": 0, "yes": 1, "no": 0, "CHURN": 1, "NOT CHURN": 0})
else:
    y = df[TARGET].astype(int)

if y.isna().any():
    raise ValueError(f"Target '{TARGET}' has unmapped values: {df[TARGET].unique()[:10]}")

print(f"Target: {TARGET}")
print(y.value_counts(normalize=True).rename("proportion").to_string())

## 3 — Feature definitions (37 kept vs 25 reduced)

In [ ]:
# Exact order used by the current backend model (37 features)
FULL_FEATURES = [
    "Gender", "Age", "Under 30", "Senior Citizen", "Married", "Dependents",
    "Number of Dependents", "Population", "Referred a Friend", "Number of Referrals",
    "Tenure in Months", "Offer", "Phone Service", "Avg Monthly Long Distance Charges",
    "Multiple Lines", "Internet Service", "Internet Type", "Avg Monthly GB Download",
    "Online Security", "Online Backup", "Device Protection Plan", "Premium Tech Support",
    "Streaming TV", "Streaming Movies", "Streaming Music", "Unlimited Data",
    "Contract", "Paperless Billing", "Payment Method", "Monthly Charge",
    "Total Charges", "Total Refunds", "Total Extra Data Charges",
    "Total Long Distance Charges", "Total Revenue", "Satisfaction Score", "CLTV",
]

# Reduced set (25) — order below is the new canonical order
REDUCED_FEATURES = [
    "Age", "Gender", "Married", "Dependents", "Tenure in Months", "Offer",
    "Phone Service", "Multiple Lines", "Internet Service", "Internet Type",
    "Online Security", "Online Backup", "Device Protection Plan", "Premium Tech Support",
    "Streaming TV", "Streaming Movies", "Streaming Music", "Unlimited Data",
    "Contract", "Paperless Billing", "Payment Method", "Monthly Charge",
    "Total Charges", "Satisfaction Score", "CLTV",
]

REMOVED_FEATURES = [c for c in FULL_FEATURES if c not in REDUCED_FEATURES]

CAT_FEATURES_REDUCE = [
    "Gender", "Married", "Dependents", "Offer", "Phone Service", "Multiple Lines",
    "Internet Service", "Internet Type", "Online Security", "Online Backup",
    "Device Protection Plan", "Premium Tech Support", "Streaming TV", "Streaming Movies",
    "Streaming Music", "Unlimited Data", "Contract", "Paperless Billing", "Payment Method",
]
NUM_FEATURES_REDUCE = [
    "Age", "Tenure in Months", "Monthly Charge", "Total Charges",
    "Satisfaction Score", "CLTV",
]

assert len(FULL_FEATURES) == 37
assert len(REDUCED_FEATURES) == 25
assert set(CAT_FEATURES_REDUCE) | set(NUM_FEATURES_REDUCE) == set(REDUCED_FEATURES)
assert len(CAT_FEATURES_REDUCE) + len(NUM_FEATURES_REDUCE) == 25

missing_full = [c for c in FULL_FEATURES if c not in df.columns]
if missing_full:
    raise KeyError(f"Dataset missing columns: {missing_full}")

print(f"Full features:    {len(FULL_FEATURES)}")
print(f"Reduced features: {len(REDUCED_FEATURES)}")
print(f"Removed:          {len(REMOVED_FEATURES)} → {REMOVED_FEATURES}")

In [ ]:
# Coerce numeric columns (Total Charges is sometimes stored as text)
for col in NUM_FEATURES_REDUCE:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["Number of Dependents", "Number of Referrals", "Avg Monthly Long Distance Charges",
            "Avg Monthly GB Download", "Total Refunds", "Total Extra Data Charges",
            "Total Long Distance Charges", "Total Revenue"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

n_before = len(df)
df = df.dropna(subset=REDUCED_FEATURES + [TARGET]).reset_index(drop=True)

# Rebuild y from the cleaned frame so index alignment is guaranteed
if df[TARGET].dtype == object:
    y = df[TARGET].map({"Yes": 1, "No": 0, "yes": 1, "no": 0})
else:
    y = pd.to_numeric(df[TARGET], errors="coerce").astype(int)

if y.isna().any():
    raise ValueError(f"Unmapped target values in {TARGET}: {df[TARGET].unique()[:10]}")

print(f"Rows: {n_before} → {len(df)} (dropped {n_before - len(df)} incomplete)")
print(y.value_counts().to_string())


## 4 — Shared split (fair comparison)

Split **once** on the full 37-feature frame, then project the same train/test rows onto the 25-feature set.

In [ ]:
X_full = df[FULL_FEATURES].copy()

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train_red = X_train_full[REDUCED_FEATURES].copy()
X_test_red = X_test_full[REDUCED_FEATURES].copy()

print(f"Train: {X_train_full.shape[0]}  Test: {X_test_full.shape[0]}")
print(f"y_train balance:\n{y_train.value_counts(normalize=True).to_string()}")
print(f"y_test  balance:\n{y_test.value_counts(normalize=True).to_string()}")

## 5 — Preprocessing + GridSearch helpers

In [ ]:
def build_preprocessor(cat_cols, num_cols):
    """Same approach as the current model: OHE(cats) + passthrough(nums) → later scaled."""
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", "passthrough", num_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


def build_pipeline(cat_cols, num_cols):
    return Pipeline([
        ("preprocessor", build_preprocessor(cat_cols, num_cols)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=42)),
    ])


PARAM_GRID = {"model__C": [0.01, 0.1, 1, 10, 100]}


def train_grid(pipeline, X_tr, y_tr):
    grid = GridSearchCV(
        pipeline,
        param_grid=PARAM_GRID,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        return_train_score=False,
    )
    grid.fit(X_tr, y_tr)
    return grid


def evaluate(grid, X_te, y_te, label):
    y_pred = grid.predict(X_te)
    metrics = {
        "model": label,
        "best_C": grid.best_params_["model__C"],
        "cv_f1": grid.best_score_,
        "accuracy": accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred, pos_label=1),
        "recall": recall_score(y_te, y_pred, pos_label=1),
        "f1": f1_score(y_te, y_pred, pos_label=1),
        "cm": confusion_matrix(y_te, y_pred, labels=[0, 1]),
        "y_pred": y_pred,
    }
    print(f"\n===== {label} =====")
    print(f"Best C: {metrics['best_C']}   CV F1 (churn): {metrics['cv_f1']:.4f}")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}  ← churn detection")
    print(f"F1       : {metrics['f1']:.4f}  ← churn detection")
    print("Confusion matrix [rows=true 0,1 | cols=pred 0,1]:")
    print(metrics["cm"])
    print(classification_report(y_te, y_pred, target_names=["Not Churn", "Churn"], digits=4))
    return metrics

## 6 — Train **baseline (37 features)**

In [ ]:
full_cats = [
    "Gender", "Under 30", "Senior Citizen", "Married", "Dependents",
    "Referred a Friend", "Offer", "Phone Service", "Multiple Lines",
    "Internet Service", "Internet Type", "Online Security", "Online Backup",
    "Device Protection Plan", "Premium Tech Support", "Streaming TV",
    "Streaming Movies", "Streaming Music", "Unlimited Data", "Contract",
    "Paperless Billing", "Payment Method",
]
full_nums = [c for c in FULL_FEATURES if c not in full_cats]

print(f"Baseline cats={len(full_cats)} nums={len(full_nums)} total={len(FULL_FEATURES)}")

grid_full = train_grid(build_pipeline(full_cats, full_nums), X_train_full, y_train)
m_full = evaluate(grid_full, X_test_full, y_test, "FULL 37-feature Logistic Regression")

## 7 — Train **reduced (25 features)**

In [ ]:
print(f"Reduced cats={len(CAT_FEATURES_REDUCE)} nums={len(NUM_FEATURES_REDUCE)} total={len(REDUCED_FEATURES)}")

grid_red = train_grid(build_pipeline(CAT_FEATURES_REDUCE, NUM_FEATURES_REDUCE), X_train_red, y_train)
m_red = evaluate(grid_red, X_test_red, y_test, "REDUCED 25-feature Logistic Regression")

## 8 — Side-by-side comparison

In [ ]:
comparison = pd.DataFrame([
    {k: m_full[k] for k in ["model", "best_C", "cv_f1", "accuracy", "precision", "recall", "f1"]},
    {k: m_red[k] for k in ["model", "best_C", "cv_f1", "accuracy", "precision", "recall", "f1"]},
])

delta = comparison.loc[1, ["accuracy", "precision", "recall", "f1"]] - comparison.loc[0, ["accuracy", "precision", "recall", "f1"]]
comparison.loc[2] = {
    "model": "Δ (reduced − full)",
    "best_C": "",
    "cv_f1": m_red["cv_f1"] - m_full["cv_f1"],
    **delta.to_dict(),
}

print("=== METRIC COMPARISON (test set) ===")
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== CONFUSION MATRICES ===")
print(f"Full ({m_full['model']}):\n{m_full['cm']}")
print(f"Reduced ({m_red['model']}):\n{m_red['cm']}")

tn_f, fp_f, fn_f, tp_f = m_full["cm"].ravel()
tn_r, fp_r, fn_r, tp_r = m_red["cm"].ravel()
print("\nChurn-class detail (class 1):")
print(f"  Full    → TP={tp_f}  FN={fn_f} (missed churners)  FP={fp_f}  TN={tn_f}")
print(f"  Reduced → TP={tp_r}  FN={fn_r} (missed churners)  FP={fp_r}  TN={tn_r}")

## 9 — Verdict (better / similar / worse)

**Acceptability rule (churn-focused):**
- **Better:** reduced churn F1 **and** recall are ≥ full model (accuracy not far behind)
- **Similar (acceptable):** churn F1 drop ≤ 0.01 **and** recall drop ≤ 0.02 **and** accuracy drop ≤ 0.01
- **Significantly worse:** any larger drop — do **not** overwrite artifacts

In [ ]:
d_acc = m_red["accuracy"] - m_full["accuracy"]
d_rec = m_red["recall"] - m_full["recall"]
d_f1  = m_red["f1"] - m_full["f1"]

print(f"Δ accuracy = {d_acc:+.4f}")
print(f"Δ recall   = {d_rec:+.4f}  (churn detection)")
print(f"Δ F1       = {d_f1:+.4f}  (churn detection)")
print(f"\nReported current model accuracy: 96.24%")
print(f"This run full-model accuracy:     {m_full['accuracy']*100:.2f}%")
print(f"This run reduced-model accuracy:  {m_red['accuracy']*100:.2f}%")

if d_f1 >= 0 and d_rec >= -0.005:
    verdict = "BETTER"
    acceptable = True
elif d_f1 >= -0.01 and d_rec >= -0.02 and d_acc >= -0.01:
    verdict = "SIMILAR (acceptable)"
    acceptable = True
else:
    verdict = "SIGNIFICANTLY WORSE"
    acceptable = False

print("\n" + "=" * 60)
print(f"VERDICT: Reduced 25-feature model is {verdict} vs the 37-feature model")
print("=" * 60)
print(f"Overwrite artifacts? {'YES' if acceptable else 'NO — keep current 37-feature pkls'}")

## 10 — Final feature list & post-preprocessing dimensions

In [ ]:
final_preprocessor = grid_red.best_estimator_.named_steps["preprocessor"]
final_model = grid_red.best_estimator_.named_steps["model"]

encoded_names = list(final_preprocessor.get_feature_names_out())

print("FINAL INPUT FEATURES (raw, 25):")
for i, f in enumerate(REDUCED_FEATURES, 1):
    kind = "cat" if f in CAT_FEATURES_REDUCE else "num"
    print(f"  {i:2d}. [{kind}] {f}")
print(f"\nRaw feature count: {len(REDUCED_FEATURES)}")

print(f"\nFEATURES AFTER PREPROCESSING (OHE + passthrough): {len(encoded_names)}")
for i, f in enumerate(encoded_names, 1):
    print(f"  {i:2d}. {f}")

assert final_model.n_features_in_ == len(encoded_names)
print(f"\nModel n_features_in_ = {final_model.n_features_in_}  ✓")
print(f"Best params: {grid_red.best_params_}")

## 11 — Save artifacts (only if acceptable)

1. Backs up the existing 37-feature `.pkl` files
2. Saves new `churn_model.pkl`, `churn_preprocessor.pkl`, `churn_scaler.pkl` **only** when verdict is acceptable

> **Note:** The FastAPI backend still validates the **37-field** schema. After replacing the pkls you must later update `backend/main.py` (and the React form) to the 25-field schema — **not done in this notebook**, per your instructions.

In [ ]:
# Where to write artifacts. In Colab, prefer downloading to your machine,
# then copy into backend/ next to main.py.
OUT_DIR = Path("artifacts_reduced_25f")
OUT_DIR.mkdir(exist_ok=True)

# Optional: point this at a mounted Drive / local backend folder to write in place
# OUT_DIR = Path("/content/drive/MyDrive/...")  or  Path(r"D:\customer churn prediction\backend")

new_artifacts = {
    "churn_model.pkl": grid_red.best_estimator_.named_steps["model"],
    "churn_preprocessor.pkl": grid_red.best_estimator_.named_steps["preprocessor"],
    "churn_scaler.pkl": grid_red.best_estimator_.named_steps["scaler"],
}

if not acceptable:
    print("Verdict = SIGNIFICANTLY WORSE → artifacts NOT saved. Current model unchanged.")
else:
    # Backup any existing pkls in OUT_DIR first
    backup_dir = OUT_DIR / "backup_37f"
    backup_dir.mkdir(exist_ok=True)
    for name in new_artifacts:
        src = OUT_DIR / name
        if src.exists():
            shutil.copy2(src, backup_dir / name)
            print(f"Backed up {name} → {backup_dir / name}")

    for name, obj in new_artifacts.items():
        path = OUT_DIR / name
        joblib.dump(obj, path)
        print(f"Saved {path} ({path.stat().st_size:,} bytes)")

    # Persist the new 25-feature contract for later API work
    meta = {
        "features": REDUCED_FEATURES,
        "categorical": CAT_FEATURES_REDUCE,
        "numeric": NUM_FEATURES_REDUCE,
        "removed": REMOVED_FEATURES,
        "best_C": grid_red.best_params_["model__C"],
        "metrics": {k: float(m_red[k]) for k in ["accuracy", "precision", "recall", "f1", "cv_f1"]},
        "n_features_after_preprocessing": len(encoded_names),
        "encoded_feature_names": encoded_names,
        "verdict": verdict,
    }
    joblib.dump(meta, OUT_DIR / "reduced_feature_meta.pkl")
    print(f"Saved {OUT_DIR / 'reduced_feature_meta.pkl'}")

In [ ]:
# Smoke-test the reduced pipeline end-to-end before you trust it
sample = X_test_red.iloc[[0]].copy()
pipe = grid_red.best_estimator_
proba = pipe.predict_proba(sample)[0, 1]
pred = int(pipe.predict(sample)[0])
print("Smoke test:")
print(sample.to_dict(orient="records")[0])
print(f"→ prediction={pred}  P(churn)={proba:.4f}")

In [ ]:
# Download artifacts to your computer (Colab)
try:
    from google.colab import files

    for p in sorted(OUT_DIR.glob("*")):
        if p.is_file():
            files.download(str(p))
    print("Downloaded artifacts — copy them into backend/ when ready.")
except Exception as e:
    print(f"Skip download ({e}). Files are in: {OUT_DIR.resolve()}")